# Día 2 · Unidad 2 — Lectura y escritura de archivos

**Objetivos de aprendizaje**
- Entender qué es un archivo, una ruta y una codificación de caracteres.
- Abrir y cerrar archivos correctamente, con el *context manager* `with`.
- Conocer los modos de apertura (`r`, `w`, `a`, `x`) y sus diferencias.
- Leer y escribir archivos de texto con los métodos habituales.

**Duración estimada:** 40-50 minutos.

**Requisitos previos:** `06_teoria_tuplas.ipynb` de este mismo día.

**Nota:** este notebook crea sus propios archivos de ejemplo al ejecutarse (`polizas_activas.txt` y `resumen_polizas.txt`, en esta misma carpeta) — no depende de ningún archivo externo.

## 1. Qué es un archivo y una ruta

Un **archivo** es un conjunto de bytes guardado en disco, con nombre y extensión (`.txt`, `.csv`, `.ipynb`...). Para acceder a él necesitas su **ruta**: dónde está dentro del sistema de carpetas.

```
Windows:  C:\Users\tu_usuario\curso-python-cac\dia_02_fundamentos\polizas_activas.txt
Unix/Mac: /home/tu_usuario/curso-python-cac/dia_02_fundamentos/polizas_activas.txt
```

Una **ruta relativa** (como `"polizas_activas.txt"` o `"./polizas_activas.txt"`) se interpreta respecto a la carpeta desde la que se ejecuta el notebook — por eso en este curso siempre trabajamos con el notebook y sus archivos de datos en la misma carpeta, o usamos `pathlib.Path` para construir la ruta explícitamente (como en `01_teoria_entorno_windows.ipynb` del Día 1).

### Codificación de caracteres

La codificación traduce bytes a caracteres legibles. Las dos más comunes son **ASCII** (128 caracteres, sin acentos ni "ñ") y **UTF-8** (más de un millón de caracteres, incluye cualquier idioma). En este curso, **especifica siempre `encoding="utf-8"`** al abrir un archivo de texto — si no lo haces, Python usa la codificación por defecto del sistema operativo, que en Windows suele ser distinta de UTF-8 y puede corromper acentos y la "ñ" al leer o escribir.

## 2. Crear un archivo de ejemplo

En vez de depender de un archivo ya existente, lo creamos aquí mismo con la magia de Jupyter `%%writefile` (escribe el contenido de la celda directamente a un archivo, sin ejecutarlo como código Python).

In [ ]:
%%writefile polizas_activas.txt
AUTO-2026-00147;Garcia, S.;480.50
HOGAR-2026-00033;Martinez, L.;275.00
AUTO-2026-00201;Fernandez, P.;510.20
VIDA-2026-00012;Lopez, R.;150.00


## 3. Abrir y cerrar archivos

Para abrir un archivo se usa `open()`. Hay que **cerrarlo siempre** después: el sistema operativo limita cuántos procesos pueden acceder a un mismo archivo a la vez, y un archivo abierto sin cerrar puede quedar bloqueado o con cambios sin guardar.

In [ ]:
f = open(file="polizas_activas.txt", mode="r", encoding="utf-8")
primera_linea = f.readline()
print(primera_linea)
f.close()   # si te olvidas de esta linea, el archivo queda abierto


La forma **correcta** de trabajar con archivos es con el *context manager* `with`: cierra el archivo automáticamente al salir del bloque, **incluso si ocurre un error dentro** — así nunca dependes de acordarte de `.close()`.

In [ ]:
with open(file="polizas_activas.txt", mode="r", encoding="utf-8") as f:
    primera_linea = f.readline()

print(primera_linea)
print("¿El archivo quedo cerrado?", f.closed)


El parámetro `mode` indica cómo se abre el archivo:

| Modo | Qué hace |
|---|---|
| `"r"` | Lectura (falla si el archivo no existe) |
| `"w"` | Escritura — **trunca** el archivo: si ya existe, borra su contenido antes de escribir |
| `"a"` | Escritura al final (*append*) — si el archivo existe, añade sin borrar lo anterior |
| `"x"` | Creación — falla si el archivo **ya** existe (evita sobrescribir por accidente) |

> ⚠️ **Error típico**: abrir un archivo que ya tiene datos importantes con `mode="w"` pensando que vas a "añadir" contenido. `"w"` **borra todo lo que hubiera antes**, sin avisar ni pedir confirmación. Si quieres añadir contenido sin perder lo existente, usa `mode="a"`.

## 4. Lectura de archivos

| Método | Qué hace |
|---|---|
| `.read()` | Lee el archivo completo (o `size` caracteres si se lo indicas) |
| `.readline()` | Lee una única línea |
| `.readlines()` | Lee todas las líneas y las devuelve como una lista de strings |

In [ ]:
with open(file="polizas_activas.txt", mode="r", encoding="utf-8") as f:
    contenido_completo = f.read()

print(contenido_completo)
print(type(contenido_completo))


In [ ]:
with open(file="polizas_activas.txt", mode="r", encoding="utf-8") as f:
    lineas = f.readlines()

print(lineas)
print(f"Numero de polizas en el archivo: {len(lineas)}")


> 📊 **En Excel esto sería...** abrir un archivo de texto delimitado por `;` con "Importar datos" o "Texto en columnas": cada línea del `.txt` es una fila, y cada campo separado por `;` sería una columna. Aquí lo hacemos "a mano" para entender el mecanismo; en el Día 7 verás que `pandas.read_csv()` hace exactamente esto en una sola línea.

In [ ]:
# Procesar cada linea: separar por ";" y quedarnos con el nombre y la prima
primas_leidas = []

with open(file="polizas_activas.txt", mode="r", encoding="utf-8") as f:
    for linea in f:
        linea = linea.strip()          # quita el salto de linea "\n" al final
        if not linea:
            continue                    # ignora lineas vacias, si las hubiera
        poliza_id, tomador, prima = linea.split(";")
        primas_leidas.append((poliza_id, tomador, float(prima)))

for registro in primas_leidas:
    print(registro)


**Para ti:** a partir de la lista `primas_leidas` de la celda anterior, calcula la prima total de la cartera (suma de la tercera posición de cada tupla) usando una list comprehension y `sum()`.

In [ ]:
# tu codigo aqui
prima_total_cartera = sum(registro[2] for registro in primas_leidas)
print(f"Prima total de la cartera: {prima_total_cartera:.2f} EUR")


## 5. Escritura de archivos

| Método | Qué hace |
|---|---|
| `.write(texto)` | Escribe un string en el archivo (no añade salto de línea automáticamente) |
| `.writelines(lista)` | Escribe una lista de strings seguidos — tampoco añade saltos de línea por ti |

In [ ]:
with open(file="resumen_polizas.txt", mode="w", encoding="utf-8") as f:
    f.write(f"Resumen de cartera - {len(primas_leidas)} polizas\n")
    f.write(f"Prima total: {prima_total_cartera:.2f} EUR\n")
    f.write("-" * 30 + "\n")
    for poliza_id, tomador, prima in primas_leidas:
        f.write(f"{poliza_id};{tomador};{prima:.2f}\n")   # el "\n" es responsabilidad tuya

print("Archivo resumen_polizas.txt escrito.")


In [ ]:
# Comprobamos leyendo lo que acabamos de escribir
with open(file="resumen_polizas.txt", mode="r", encoding="utf-8") as f:
    print(f.read())


> ⚠️ **Error típico**: olvidar el `"\n"` al final de cada `.write()`. A diferencia de `print()`, que añade un salto de línea automáticamente, `.write()` **no lo hace** — si escribes varias líneas seguidas sin `"\n"`, terminan todas pegadas en una sola línea del archivo.

In [ ]:
# Anadimos una poliza nueva sin borrar lo que ya habia, con mode="a"
with open(file="resumen_polizas.txt", mode="a", encoding="utf-8") as f:
    f.write("VIDA-2026-00099;Ruiz, A.;95.00\n")

with open(file="resumen_polizas.txt", mode="r", encoding="utf-8") as f:
    print(f.read())


**Para ti:** crea un archivo `polizas_impagadas.txt` (con `mode="w"`) que contenga, una por línea, las pólizas de `primas_leidas` con prima superior a `300`. Después ábrelo y muestra su contenido para comprobarlo.

In [ ]:
# tu codigo aqui
with open(file="polizas_impagadas.txt", mode="w", encoding="utf-8") as f:
    for poliza_id, tomador, prima in primas_leidas:
        if prima > 300:
            f.write(f"{poliza_id};{tomador};{prima:.2f}\n")

with open(file="polizas_impagadas.txt", mode="r", encoding="utf-8") as f:
    print(f.read())


## Resumen

- Una ruta relativa se interpreta desde la carpeta en la que se ejecuta el notebook; especifica siempre `encoding="utf-8"` al abrir texto.
- Usa siempre `with open(...) as f:` en vez de `open()`/`.close()` sueltos — cierra el archivo automáticamente, incluso si hay un error dentro del bloque.
- Modos de apertura: `"r"` lectura, `"w"` escritura (**trunca** el archivo existente), `"a"` añade al final sin borrar, `"x"` crea y falla si ya existe.
- Lectura: `.read()` (todo de una vez), `.readline()` (una línea), `.readlines()` (lista de líneas) — o iterar directamente con `for linea in f:`.
- Escritura: `.write()` y `.writelines()` no añaden saltos de línea por ti — tienes que incluir `"\n"` explícitamente.
- Procesar un archivo de texto delimitado (`;`, `,`) a mano con `.split()` es exactamente lo que automatiza `pandas.read_csv()`, que verás en el Día 7.

**Siguiente:** `08_ejercicios.ipynb` — practica todo lo visto en este día antes de pasar a `09_soluciones.ipynb`.